In [15]:
import numpy as np
import pandas as pd
import ast
import pickle

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [17]:
from nltk.stem.porter import PorterStemmer


In [18]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv', engine='python', on_bad_lines='skip')

In [19]:
movies = movies.merge(credits, on='title')

In [68]:
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

In [69]:
movies.dropna(inplace=True)

In [73]:
import ast

def convert(text):

    L = []

    # if already list
    if isinstance(text, list):

        # check if list contains dictionaries
        if len(text) > 0 and isinstance(text[0], dict):

            for i in text:
                L.append(i['name'])

            return L

        return text

    if isinstance(text, str):

        try:
            for i in ast.literal_eval(text):
                L.append(i['name'])

            return L

        except:
            return []

    return []


In [74]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [71]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[[, {, "", i, d, "", :, , 1, 4, 6, 3, ,, , "", n,...",[],[]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[[, {, "", i, d, "", :, , 2, 7, 0, ,, , "", n, a,...",[],[]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[[, {, "", i, d, "", :, , 4, 7, 0, ,, , "", n, a,...",[],[]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[[, {, "", i, d, "", :, , 8, 4, 9, ,, , "", n, a,...",[],[]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[[, {, "", i, d, "", :, , 8, 1, 8, ,, , "", n, a,...",[],[]


In [77]:
def convert_cast(text):

    L = []

    if isinstance(text, list):

        if len(text) > 0 and isinstance(text[0], dict):

            counter = 0

            for i in text:

                if counter != 3:
                    L.append(i['name'])
                    counter += 1
                else:
                  break

            return L

        return text

    if isinstance(text, str):

        try:
            counter = 0

            for i in ast.literal_eval(text):

                if counter != 3:
                    L.append(i['name'])
                    counter += 1
                else:
                    break

            return L

        except:
            return []

    return []

movies['cast'] = movies['cast'].apply(convert_cast)

In [88]:
def fetch_director(text):

    L = []

    if isinstance(text, list):

        if len(text) > 0 and isinstance(text[0], dict):

            for i in text:

                if i['job'] == 'Director':
                    L.append(i['name'])

            return L

        return text

    if isinstance(text, str):

        try:
            for i in ast.literal_eval(text):

                if i['job'] == 'Director':
                    L.append(i['name'])

            return L

        except:
            return []

    return []

movies['crew'] = movies['crew'].apply(fetch_director)

In [97]:
movies['overview'] = movies['overview'].apply(
    lambda x: x if isinstance(x, list) else str(x).split()
)

In [98]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [100]:
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['crew']
)

In [101]:
new_df = movies[['movie_id', 'title', 'tags']]

In [102]:
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x))


/tmp/ipykernel_530/2557390870.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x))


In [103]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

/tmp/ipykernel_530/1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [104]:
ps = PorterStemmer()


def stem(text):

    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return ' '.join(y)


new_df['tags'] = new_df['tags'].apply(stem)


/tmp/ipykernel_530/458188941.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [105]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = cv.fit_transform(new_df['tags']).toarray()

In [106]:
similarity = cosine_similarity(vectors)

In [108]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print(f'\n🎬 Recommendations for {movie}:\n')

    for i in movies_list:
        print(new_df.iloc[i[0]].title)


In [109]:
recommend('Batman Begins')


🎬 Recommendations for Batman Begins:

Synecdoche, New York
The Shipping News
Sexy Beast
10th & Wolf
Copying Beethoven


In [111]:
pickle.dump(
    new_df.to_dict(),
    open('movies_dict.pkl', 'wb')
)

pickle.dump(
    similarity,
    open('similarity.pkl', 'wb')
)

print('\n✅ PKL files created successfully!')


✅ PKL files created successfully!
